# Member 3: Feature Engineering + Random Forest

**Loan Default Prediction System**

Team Member 3 Tasks:
- Feature Engineering
- Random Forest Algorithm

In [10]:
# STEP 2: Loading the Dataset
import pandas as pd
import numpy as np

# Load the dataset - Change the filename if yours is different
df = pd.read_csv('../data/credit_risk_dataset.csv')

# Basic information
print("=== Dataset Shape ===")
print(df.shape)

print("\n=== First 5 Rows ===")
print(df.head())

print("\n=== Column Names ===")
print(df.columns.tolist())

print("\n=== Data Types ===")
print(df.dtypes)

print("\n=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Target Variable (loan_status) Distribution ===")
print(df['loan_status'].value_counts(normalize=True) * 100)

=== Dataset Shape ===
(32581, 12)

=== First 5 Rows ===
   person_age  person_income person_home_ownership  person_emp_length  \
0          22          59000                  RENT              123.0   
1          21           9600                   OWN                5.0   
2          25           9600              MORTGAGE                1.0   
3          23          65500                  RENT                4.0   
4          24          54400                  RENT                8.0   

  loan_intent loan_grade  loan_amnt  loan_int_rate  loan_status  \
0    PERSONAL          D      35000          16.02            1   
1   EDUCATION          B       1000          11.14            0   
2     MEDICAL          C       5500          12.87            1   
3     MEDICAL          C      35000          15.23            1   
4     MEDICAL          C      35000          14.27            1   

   loan_percent_income cb_person_default_on_file  cb_person_cred_hist_length  
0                 0.59 

## Step 3: Feature Engineering Plan

### Why Feature Engineering?
- Raw data has limitations.
- We create new useful columns (features) so Random Forest can learn better patterns for predicting who will default on a loan.
- Good features = higher accuracy, better precision and recall = higher marks.

### Planned New Features:

1. **Debt-to-Income Ratio (DTI)**  
   `dti_ratio = loan_amnt / person_income`  
   Why: Shows how big the loan is compared to income. Higher = more risky.

2. **Loan-to-Income Ratio**  
   Similar to DTI, helps the model understand loan burden.

3. **Age Group**  
   Convert `person_age` into categories: Young (18-25), Adult (26-35), Middle (36-50), Senior (51+)

4. **Income Group**  
   Categorize `person_income` into Low, Medium, High.

5. **Interest Rate Category**  
   `loan_int_rate` → Low, Medium, High interest groups.

6. **Loan Amount Category**  
   `loan_amnt` → Small, Medium, Large loans.

7. **Employment Length Category**  
   `person_emp_length` → Newbie, Experienced, Veteran.

### Next Steps:
We will create these features one by one using pandas.

In [11]:
# STEP 4: Creating First Engineered Feature - Debt-to-Income Ratio (DTI)

# Create Debt-to-Income Ratio
df['dti_ratio'] = df['loan_amnt'] / df['person_income']

# Check the new feature
print("=== New Feature: dti_ratio ===")
print(df[['loan_amnt', 'person_income', 'dti_ratio']].head(10))

print("\n=== Basic Statistics of dti_ratio ===")
print(df['dti_ratio'].describe())

# Optional: Check correlation with target (how well it relates to default)
print("\n=== Correlation with loan_status (Target) ===")
print(df[['dti_ratio', 'loan_status']].corr())

=== New Feature: dti_ratio ===
   loan_amnt  person_income  dti_ratio
0      35000          59000   0.593220
1       1000           9600   0.104167
2       5500           9600   0.572917
3      35000          65500   0.534351
4      35000          54400   0.643382
5       2500           9900   0.252525
6      35000          77100   0.453956
7      35000          78956   0.443285
8      35000          83000   0.421687
9       1600          10000   0.160000

=== Basic Statistics of dti_ratio ===
count    32581.000000
mean         0.170553
std          0.107049
min          0.000789
25%          0.089655
50%          0.148148
75%          0.229167
max          0.830000
Name: dti_ratio, dtype: float64

=== Correlation with loan_status (Target) ===
             dti_ratio  loan_status
dti_ratio     1.000000     0.385873
loan_status   0.385873     1.000000


In [12]:
# STEP 5: Creating Age Group and Income Group Features

# 1. Age Group
def get_age_group(age):
    if age <= 25:
        return 'Young'
    elif age <= 35:
        return 'Adult'
    elif age <= 50:
        return 'Middle-aged'
    else:
        return 'Senior'

df['age_group'] = df['person_age'].apply(get_age_group)

# 2. Income Group
def get_income_group(income):
    if income <= 30000:
        return 'Low'
    elif income <= 60000:
        return 'Medium'
    elif income <= 100000:
        return 'High'
    else:
        return 'Very High'

df['income_group'] = df['person_income'].apply(get_income_group)

# Check the new features
print("=== New Features: age_group and income_group ===")
print(df[['person_age', 'age_group', 'person_income', 'income_group']].head(10))

print("\n=== Distribution of Age Groups ===")
print(df['age_group'].value_counts())

print("\n=== Distribution of Income Groups ===")
print(df['income_group'].value_counts())

# Optional: See default rate by groups (very useful for analysis)
print("\n=== Default Rate by Age Group ===")
print(df.groupby('age_group')['loan_status'].mean() * 100)

print("\n=== Default Rate by Income Group ===")
print(df.groupby('income_group')['loan_status'].mean() * 100)

=== New Features: age_group and income_group ===
   person_age age_group  person_income income_group
0          22     Young          59000       Medium
1          21     Young           9600          Low
2          25     Young           9600          Low
3          23     Young          65500         High
4          24     Young          54400       Medium
5          21     Young           9900          Low
6          26     Adult          77100         High
7          24     Young          78956         High
8          24     Young          83000         High
9          21     Young          10000          Low

=== Distribution of Age Groups ===
age_group
Young          15352
Adult          13763
Middle-aged     3178
Senior           288
Name: count, dtype: int64

=== Distribution of Income Groups ===
income_group
Medium       14210
High          9648
Low           4516
Very High     4207
Name: count, dtype: int64

=== Default Rate by Age Group ===
age_group
Adult          20.678631